# 04 — Experiments

Runs the backbone screen and the five required experiments, then reports the
results with significance tests.

**Every run is the same `ExperimentConfig` with different field values.** There
is one training loop in `src/engine.py`; the arms differ by configuration, never
by code path. Divergent per-variant scripts drift, and a drifted baseline
invalidates every comparison made here.

Training is expensive. Both phases below are **resumable**: `run_matrix` skips
any run already present in `results/summary.csv`, so re-executing a cell after an
interruption costs only the unfinished runs.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch

from src import config, engine, experiments, metrics, splits, stats, viz

viz.apply_style()
pd.set_option("display.width", 200)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

## Phase 1 — backbone screen

Four backbones, 6 epochs, identical hierarchical configuration, one seed.

This selects the backbone for Phase 2. It is **not** the backbone comparison
result: six epochs need not rank backbones the way a full run would, and the
write-up should present it as a selection step with that limitation stated.

In [ ]:
# Skips anything already in results/summary.csv.
screen = experiments.run_matrix(experiments.screen_configs(epochs=6))

In [ ]:
summary = pd.read_csv(engine.RESULTS_DIR / "summary.csv")
scr = summary[summary.arm.str.startswith("screen_")].copy()

cols = ["backbone", "val_macro_f1", "test_macro_f1", "test_balanced_accuracy",
        "test_cross_lineage_error", "total_minutes"]
scr[cols].sort_values("val_macro_f1", ascending=False).reset_index(drop=True)

In [ ]:
# Selection is on validation macro F1 - never test, which stays untouched
# until the final evaluation, and never accuracy.
BEST_BACKBONE = scr.sort_values("val_macro_f1", ascending=False).iloc[0]["backbone"]
print(f"selected backbone: {BEST_BACKBONE}")

## Phase 2 — the five required experiments

Four distinct configurations at three seeds each = 12 runs, all on the selected
backbone with identical splits.

Seeds provide the paired samples for the significance tests. CLAUDE.md asks for a
paired t-test "across validation folds", but `splits.py` produces one fixed split
and there are no folds — and that split must stay identical across arms for the
ablations to mean anything. Retraining at several seeds pairs the arms while
holding the partition fixed.

In [ ]:
main = experiments.run_matrix(
    experiments.main_configs(backbone=BEST_BACKBONE, epochs=20)
)

In [ ]:
summary = pd.read_csv(engine.RESULTS_DIR / "summary.csv")
runs = summary[~summary.arm.str.startswith(("screen_", "smoke_"))].copy()
print(f"{len(runs)} runs across {runs.arm.nunique()} arms and {runs.seed.nunique()} seeds")

arm_stats = stats.arm_summary(runs)
arm_stats.round(4)

### Headline comparison

Macro F1 and balanced accuracy are the headline metrics, not accuracy. At 260:1
imbalance a model that ignores reactive lymphocytes entirely loses ~0.08%
accuracy, so accuracy cannot see the behaviour this dissertation is about.

In [ ]:
f = viz.arm_comparison(arm_stats, metric="test_macro_f1",
                       title="Test macro F1 by arm (mean ± sd over seeds)")
f.savefig(config.ARTIFACT_DIR / "arm_macro_f1.png")

f = viz.arm_comparison(arm_stats, metric="test_minority_macro_f1",
                       title="Test macro F1, minority classes only")
f.savefig(config.ARTIFACT_DIR / "arm_minority_f1.png")

### Hierarchical error composition

The figure that carries the hierarchical claim. Errors are split into
within-lineage (clinically mild — typically adjacent maturation stages) and
cross-lineage (clinically severe).

A lineage-aware model should shrink the cross-lineage segment specifically,
moving error toward the milder kind. That shift can happen with little or no
change in overall accuracy, which is exactly why accuracy is the wrong headline.

In [ ]:
f = viz.error_composition(arm_stats,
                          title="Prediction composition by arm (test set)")
f.savefig(config.ARTIFACT_DIR / "error_composition.png")

### Significance

Paired t-tests across seeds against the flat baseline, with Cohen's d.

Read these with the sample size in mind: three seeds gives 2 degrees of freedom
and very little power. A non-significant result here is **weak evidence of no
difference, not evidence of no difference**. Effect sizes and per-seed values are
reported alongside so the conclusion never rests on the p-value alone.

In [ ]:
comp = stats.comparison_table(runs, baseline="flat_baseline", metric="test_macro_f1")
for _, r in comp.iterrows():
    print(stats.format_result(r.to_dict()))
print()
comp.round(4)

In [ ]:
# The same tests on the minority-class metric and on cross-lineage error -
# the two quantities the dissertation actually argues about.
for metric in ("test_minority_macro_f1", "test_cross_lineage_error",
               "test_balanced_accuracy"):
    print(f"--- {metric} ---")
    for _, r in stats.comparison_table(runs, "flat_baseline", metric).iterrows():
        print("  " + stats.format_result(r.to_dict()))
    print()

### Per-seed values

Shown in full rather than summarised, so the spread behind each mean is visible
and the reader can judge the t-tests for themselves.

In [ ]:
runs.pivot_table(index="arm", columns="seed", values="test_macro_f1").round(4)

### Training curves

Loss components, the selection metric, and how the error split evolves. If the
hierarchy is contributing, the lineage and consistency curves must actually move
— a flat consistency curve would mean the term is inert.

In [ ]:
run_id = runs[(runs.arm == "hierarchical") & (runs.seed == 0)].iloc[0]["run_id"]
hist = pd.read_csv(engine.RESULTS_DIR / f"history_{run_id}.csv")

f = viz.training_curves(hist, title=f"Training: {run_id}")
f.savefig(config.ARTIFACT_DIR / "training_curves.png")

---

## Summary

Results are written to `results/summary.csv` (one row per run) and
`results/history_*.csv` (per-epoch). Notebook 05 does the detailed evaluation:
confusion matrices, per-class behaviour, and Grad-CAM.